# FUNCTIONS

In [1]:
import pandas as pd
import yfinance as yf


def download_data(
    index,
    components,
    start_date,
    end_date
):
    """
    Download index and component price data.

    Failed/unavailable tickers are skipped automatically.
    The returned DataFrame contains only successfully
    downloaded tickers.
    """

    tickers = [index] + list(components)

    downloaded = {}
    failed = []

    for ticker in tickers:

        try:
            df = yf.download(
                ticker,
                start=start_date,
                end=end_date,
                auto_adjust=False,
                progress=False,
                threads=False
            )

            # Check whether data was actually returned
            if df is None or df.empty:
                failed.append(ticker)
                continue

            # Handle yfinance MultiIndex columns
            if isinstance(df.columns, pd.MultiIndex):
                if "Close" in df.columns.get_level_values(0):
                    close = df["Close"]

                    if isinstance(close, pd.DataFrame):
                        close = close.iloc[:, 0]

                else:
                    failed.append(ticker)
                    continue

            else:
                if "Close" not in df.columns:
                    failed.append(ticker)
                    continue

                close = df["Close"]

            # Remove completely missing observations
            close = close.dropna()

            if close.empty:
                failed.append(ticker)
                continue

            downloaded[ticker] = close

        except Exception:
            failed.append(ticker)

    if not downloaded:
        raise ValueError("No ticker could be downloaded.")

    # Build DataFrame in ONE operation
    data = pd.concat(
        downloaded,
        axis=1
    )

    data.columns.name = None

    # Remove completely empty columns
    data = data.dropna(axis=1, how="all")

    # Make sure index is sorted
    data = data.sort_index()

    # Optional information about failed tickers
    if failed:
        print(
            f"Skipped {len(failed)} unavailable ticker(s): "
            + ", ".join(failed)
        )

    return data

In [2]:
def add_daily_returns(data):
    """
    Add daily percentage returns for all price columns.
    """

    return_columns = {
        f"{column}_return": data[column].pct_change()
        for column in data.columns
    }

    return pd.concat(
        [data, pd.DataFrame(return_columns, index=data.index)],
        axis=1
    )

In [3]:
def add_rolling_mean(data, columns, window=21):
    if isinstance(columns, str):
        columns = [col for col in data.columns if columns in col]

    for column in columns:
        data[f"{column}_ma{window}"] = data[column].rolling(window).mean()

    return data

In [4]:
def add_rolling_stats(data, window=21):
    """
    Add rolling statistics and excess-return statistics.

    For every return column:
        <name>_mean
        <name>_sd
        <name>_mean_sd

    For every component (excluding INDEX):
        <name>_r_excess
        <name>_mu_excess
        <name>_sd_excess
        <name>_Z_excess

    Uses pd.concat() to avoid DataFrame fragmentation.
    """

    return_cols = [
        col for col in data.columns
        if col.endswith("_return")
    ]

    index_return = "INDEX_return"

    if index_return not in data.columns:
        raise ValueError(
            f"'{index_return}' not found in data. "
            "The index return column is required."
        )

    new_columns = {}

    for col in return_cols:

        name = col[:-len("_return")]

        # -----------------------------------------------------
        # Rolling statistics
        # -----------------------------------------------------
        rolling = data[col].rolling(window)

        mean = rolling.mean()
        sd = rolling.std()

        new_columns[f"{name}_mean"] = mean
        new_columns[f"{name}_sd"] = sd
        new_columns[f"{name}_mean_sd"] = mean / sd

        # -----------------------------------------------------
        # Excess return statistics
        # -----------------------------------------------------
        if name != "INDEX":

            r_excess = (
                data[col] - data[index_return]
            )

            mu_excess = (
                r_excess.rolling(window).mean()
            )

            sd_excess = (
                r_excess.rolling(window).std()
            )

            Z_excess = (
                mu_excess / sd_excess
            )

            new_columns[f"{name}_r_excess"] = r_excess
            new_columns[f"{name}_mu_excess"] = mu_excess
            new_columns[f"{name}_sd_excess"] = sd_excess
            new_columns[f"{name}_Z_excess"] = Z_excess

    # ---------------------------------------------------------
    # Add everything in ONE operation
    # ---------------------------------------------------------
    new_data = pd.DataFrame(
        new_columns,
        index=data.index
    )

    return pd.concat(
        [data, new_data],
        axis=1
    )

In [5]:
import pandas as pd
from datetime import date


def create_dow_membership(end_date=None):
    """
    Create point-in-time DJIA component membership.

    Columns:
        ticker : ticker symbol
        from   : first date as a DJIA component
        to     : last date as a DJIA component

    If end_date is None, today's system date is used.
    """

    # --------------------------------------------------
    # Dynamic end date
    # --------------------------------------------------
    if end_date is None:
        end_date = pd.Timestamp(date.today())
    else:
        end_date = pd.Timestamp(end_date)

    # --------------------------------------------------
    # Historical DJIA membership
    # Start = 2000-01-01
    # --------------------------------------------------
    records = [

        # ==================================================
        # Components already in the Dow on 2000-01-01
        # ==================================================

        ("AA",   "2000-01-01", "2013-09-23"),
        ("AXP",  "2000-01-01", end_date),
        ("BA",   "2000-01-01", end_date),
        ("C",    "2000-01-01", "2009-06-08"),
        ("CAT",  "2000-01-01", end_date),
        ("DD",   "2000-01-01", "2017-09-01"),
        ("DIS",  "2000-01-01", end_date),
        ("EK",   "2000-01-01", "2004-04-08"),
        ("GE",   "2000-01-01", "2018-06-26"),
        ("GM",   "2000-01-01", "2009-06-08"),
        ("HD",   "2000-01-01", end_date),
        ("HON",  "2000-01-01", "2008-02-19"),
        ("HPQ",  "2000-01-01", "2013-09-23"),
        ("IBM",  "2000-01-01", end_date),
        ("INTC", "2000-01-01", "2024-11-08"),
        ("IP",   "2000-01-01", "2004-04-08"),
        ("JNJ",  "2000-01-01", end_date),
        ("JPM",  "2000-01-01", end_date),
        ("KO",   "2000-01-01", end_date),
        ("MCD",  "2000-01-01", end_date),
        ("MMM",  "2000-01-01", end_date),
        ("MO",   "2000-01-01", "2008-02-19"),
        ("MRK",  "2000-01-01", end_date),
        ("MSFT", "2000-01-01", end_date),
        ("PG",   "2000-01-01", end_date),
        ("T",    "2000-01-01", "2015-03-19"),
        ("UTX",  "2000-01-01", "2020-04-06"),
        ("WMT",  "2000-01-01", end_date),
        ("XOM",  "2000-01-01", "2020-08-31"),

        # ==================================================
        # 2004
        # AIG, Pfizer, Verizon replace AT&T, Kodak,
        # International Paper
        # ==================================================

        ("AIG",  "2004-04-08", "2008-09-22"),
        ("PFE",  "2004-04-08", "2020-08-31"),
        ("VZ",   "2004-04-08", "2026-06-29"),

        # ==================================================
        # 2008-02-19
        # Chevron + Bank of America replace
        # Altria + Honeywell
        # ==================================================

        ("CVX",  "2008-02-19", end_date),
        ("BAC",  "2008-02-19", "2013-09-23"),

        # ==================================================
        # 2008-09-22
        # Kraft replaces AIG
        # ==================================================

        ("KFT",  "2008-09-22", "2012-09-24"),

        # ==================================================
        # 2009-06-08
        # Travelers + Cisco replace GM + Citigroup
        # ==================================================

        ("TRV",  "2009-06-08", end_date),
        ("CSCO", "2009-06-08", end_date),

        # ==================================================
        # 2012-09-24
        # UnitedHealth replaces Kraft
        # ==================================================

        ("UNH",  "2012-09-24", end_date),

        # ==================================================
        # 2013-09-23
        # Goldman Sachs + Nike + Visa replace
        # Alcoa + Bank of America + Hewlett-Packard
        # ==================================================

        ("GS",   "2013-09-23", end_date),
        ("NKE",  "2013-09-23", end_date),
        ("V",    "2013-09-23", end_date),

        # ==================================================
        # 2015-03-19
        # Apple replaces AT&T
        # ==================================================

        ("AAPL", "2015-03-19", end_date),

        # ==================================================
        # 2017-09-01
        # DowDuPont replaces DuPont
        # ==================================================

        ("DWDP", "2017-09-01", "2019-04-02"),

        # ==================================================
        # 2018-06-26
        # Walgreens replaces General Electric
        # ==================================================

        ("WBA",  "2018-06-26", "2024-02-26"),

        # ==================================================
        # 2019-04-02
        # Dow Inc. replaces DowDuPont
        # ==================================================

        ("DOW",  "2019-04-02", "2024-11-08"),

        # ==================================================
        # 2020-04-06
        # Raytheon Technologies replaces United Technologies
        # ==================================================

        ("RTX",  "2020-04-06", "2020-08-31"),

        # ==================================================
        # 2020-08-31
        # Amgen + Honeywell + Salesforce replace
        # ExxonMobil + Pfizer + Raytheon Technologies
        # ==================================================

        ("AMGN", "2020-08-31", end_date),
        ("CRM",  "2020-08-31", end_date),

        # Honeywell was removed in 2008 and re-added in 2020
        ("HON",  "2020-08-31", end_date),

        # ==================================================
        # 2024-02-26
        # Amazon replaces Walgreens
        # ==================================================

        ("AMZN", "2024-02-26", end_date),

        # ==================================================
        # 2024-11-08
        # Nvidia replaces Intel
        # Sherwin-Williams replaces Dow Inc.
        # ==================================================

        ("NVDA", "2024-11-08", end_date),
        ("SHW",  "2024-11-08", end_date),

        # ==================================================
        # 2026-06-29
        # Alphabet replaces Verizon
        # ==================================================

        ("GOOGL", "2026-06-29", end_date),
    ]

    # --------------------------------------------------
    # Create DataFrame
    # --------------------------------------------------

    dow_membership = pd.DataFrame(
        records,
        columns=["ticker", "from", "to"]
    )

    dow_membership["from"] = pd.to_datetime(
        dow_membership["from"]
    )

    dow_membership["to"] = pd.to_datetime(
        dow_membership["to"]
    )

    # --------------------------------------------------
    # Remove records that start after requested end_date
    # --------------------------------------------------

    dow_membership = dow_membership[
        dow_membership["from"] <= end_date
    ].copy()

    # Make sure no "to" goes beyond end_date
    dow_membership["to"] = dow_membership["to"].clip(
        upper=end_date
    )

    # --------------------------------------------------
    # Sort
    # --------------------------------------------------

    dow_membership = (
        dow_membership
        .sort_values(["from", "ticker"])
        .reset_index(drop=True)
    )

    return dow_membership

In [6]:
def prepare_data(
    index,
    components,
    start_date,
    end_date,
    rolling_window=21,
    ma_window=63
):
    """
    Prepare data for the trading model.
    """

    # 1. Download
    data = download_data(
        index,
        components,
        start_date,
        end_date
    )

    if data is None or data.empty:
        raise ValueError("No data was downloaded.")

    # The rest of the code expects the index to be called INDEX.
    data = data.rename(
        columns={index: "INDEX"}
    )

    # 2. Daily returns
    data = add_daily_returns(data)

    # 3. Rolling statistics + excess statistics
    data = add_rolling_stats(
        data,
        window=rolling_window
    )

    # 4. Moving average of Z_excess
    data = add_rolling_mean(
        data,
        "_Z_excess",
        window=ma_window
    )

    return data.copy()

# DATA

In [7]:
dow_membership = create_dow_membership()

In [ ]:
# TRAIN DATA
data_df = prepare_data(
    "^DJI",
    dow_membership["ticker"].tolist(),
    "2000-01-01",
    "2023-12-31"
)

In [9]:
import os

os.makedirs("../data", exist_ok=True)

data_df.to_parquet(
    "../data/data_df.parquet"
)

dow_membership.to_csv(
    "../data/dow_membership.csv",
    index=False
)